# Week 10 — Wednesday: Ridge and Lasso

**DATA 202 · Calvin University**

Monday's line priced Ames houses from 47 columns: R² 0.87 in cross-validation, and no overfitting.

> 🎯 **What happens when we give a line more freedom — and how do we take some of it back?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~8 min | 1 · More freedom: products of columns |
| ~7 min | 2 · Trillion-dollar coefficients |
| ~10 min | 3 · Ridge: a price on big coefficients (SLO 10B) |
| ~7 min | 4 · Choosing alpha (SLO 10B) |
| ~8 min | 5 · Lasso: some coefficients go to zero (SLO 10B) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: De Cock (2011), *Journal of Statistics Education* 19(3); from the Ames City Assessor's Office</sub>

---
## 1 · More Freedom: Products of Columns · ~8 min

Monday's model gives each column one fixed slope: $143 per sq ft in every house. But does an extra square foot add the same in a run-down house as in a well-built one?

**Product columns:** add `living_area × quality`, `living_area × year_built`, … every pair of numeric columns, plus each one squared.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

ames = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/ames.csv")
X = ames.drop(columns="price")
y = ames["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)    # Monday's split
numeric = [c for c in X.columns if c not in ("neighborhood", "year_sold")]

def house_model(regressor, products):
    """Monday's pipeline; products=True adds every product of two numeric columns."""
    num_steps = [SimpleImputer(strategy="median"), StandardScaler()]
    if products:
        num_steps.append(PolynomialFeatures(degree=2, include_bias=False))
    prep = ColumnTransformer([("num", make_pipeline(*num_steps), numeric),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), ["neighborhood"])])
    return Pipeline([("prep", prep), ("regressor", regressor)])

rows = []
for name, products in [("Monday: 47 columns", False), ("with products: 237 columns", True)]:
    scores = cross_validate(house_model(LinearRegression(), products), X_train, y_train, cv=5,
                            scoring="neg_root_mean_squared_error", return_train_score=True)
    rows.append({"model": name, "RMSE, folds it learned from": -scores["train_score"].mean(),
                 "RMSE, held-out fold": -scores["test_score"].mean()})
pd.DataFrame(rows).round(0)

> 📄 **Handout 1 — More freedom.** Fill in the gaps between the two columns. Which model is overfitting?

<details><summary>Answer</summary>

* **Monday:** $27,069 vs. $27,876. A gap of about $800: no overfitting
* **With products:** $19,548 vs. $28,958. The misses on known houses shrink by $7,500, but the misses on new houses **grow**: **overfitting**
* The scaler is back (Week 9): products of raw square feet and years would be enormous numbers
</details>

---
## 2 · Trillion-Dollar Coefficients · ~7 min

In [ ]:
ols = house_model(LinearRegression(), products=True).fit(X_train, y_train)
coefs = pd.Series(ols.named_steps["regressor"].coef_, index=ols.named_steps["prep"].get_feature_names_out())
coefs.abs().sort_values(ascending=False).head(5).apply(lambda v: f"${v:,.0f}")

> 💬 **Question:** Coefficients of trillions of dollars per standard deviation (more than all the houses in Iowa are worth), yet the predictions are fine. How?

<details><summary>Answer</summary>

* `living_area` = `first_floor_area` + `second_floor_area`, exactly. Add a trillion to one coefficient and subtract it from the other two: same predictions
* With 237 columns, many such combinations exist. Least squares can't choose among them, so it picks wild ones that cancel out on the training houses, and may stop canceling on a new house
* Picture the squared error as a landscape over two coefficients: a long flat valley, with no single bottom
</details>

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/10/images/ridge_surface.png" width="640" alt="Two 3-D surfaces of squared error over two coefficients, beta1 and beta2. Left, least squares: a long flat trough, labeled unstable. Right, ridge: a bowl with a single lowest point, labeled stable.">

---
## 3 · Ridge: a Price on Big Coefficients (SLO 10B) · ~10 min

**Ridge** adds a cost for every coefficient's size to the least-squares error:

$$\text{minimize} \quad \sum (\text{actual} - \text{predicted})^2 \; + \; \alpha \sum b_j^2$$

* `alpha` = 0: plain least squares. A bigger `alpha` pushes every coefficient toward 0
* The valley gets a bottom (right picture): one best set of coefficients
* The penalty compares coefficients with each other, so the columns must share one ruler: **scale first**

In [ ]:
ridge = house_model(Ridge(alpha=300), products=True).fit(X_train, y_train)
ridge_coefs = pd.Series(ridge.named_steps["regressor"].coef_, index=ridge.named_steps["prep"].get_feature_names_out())
ridge_coefs.abs().sort_values(ascending=False).head(5).apply(lambda v: f"${v:,.0f}")

<details><summary>Answer</summary>

* The largest coefficient is now **$13,919**, for `quality`, followed by living area, basement area and the years: numbers a person can read
* Same 237 columns, all still in the model, just smaller
</details>

---
## 4 · Choosing alpha (SLO 10B) · ~7 min

`alpha` is a knob, like `k` and `max_depth`. Choose it with cross-validation, on the training houses only.

In [ ]:
alphas = [0.1, 1, 10, 30, 100, 300, 1000, 3000, 10000]
ridge_search = GridSearchCV(house_model(Ridge(), products=True), {"regressor__alpha": alphas}, cv=5,
                            scoring="neg_root_mean_squared_error", return_train_score=True)
ridge_search.fit(X_train, y_train)

curve = pd.DataFrame({"alpha": alphas,
                      "folds it learned from": -ridge_search.cv_results_["mean_train_score"],
                      "held-out fold": -ridge_search.cv_results_["mean_test_score"]})
fig = px.line(curve, x="alpha", y=["folds it learned from", "held-out fold"], markers=True, log_x=True,
              labels={"value": "RMSE ($)", "variable": "scored on"}, title="Ridge: Choosing alpha by 5-Fold CV")
fig.show()

> 📄 **Handout 2 — Choose alpha.** Circle the best alpha. Label the overfitting side and the underfitting side.

<details><summary>Answer</summary>

* **Best: alpha = 300**, held-out RMSE **$25,106** (least squares with products: $28,958; Monday: $27,876)
* **Small alpha:** small misses on known houses, bigger on new ones → **overfitting**
* **alpha = 10,000:** about $39,000 on both → **underfitting**: the penalty flattens every slope
* Same picture as k in kNN and depth in trees, in a new costume: this time a **larger** knob means a **simpler** model
</details>

---
## 5 · Lasso: Some Coefficients Go to Zero (SLO 10B) · ~8 min

**Lasso** charges for the coefficients' absolute sizes instead:

$$\text{minimize} \quad \sum (\text{actual} - \text{predicted})^2 \; + \; \alpha \sum |b_j|$$

Same idea, one difference: Lasso pushes many coefficients **all the way to 0**. Those columns drop out of the model.

In [ ]:
lasso_alphas = [10, 30, 100, 300, 1000, 3000, 10000]
lasso_search = GridSearchCV(house_model(Lasso(max_iter=200_000), products=True), {"regressor__alpha": lasso_alphas},
                            cv=5, scoring="neg_root_mean_squared_error")
lasso_search.fit(X_train, y_train)

kept = []
for alpha in lasso_alphas:
    fit = house_model(Lasso(alpha=alpha, max_iter=200_000), products=True).fit(X_train, y_train)
    kept.append((fit.named_steps["regressor"].coef_ != 0).sum())
pd.DataFrame({"alpha": lasso_alphas, "columns kept (of 237)": kept,
              "CV RMSE ($)": -lasso_search.cv_results_["mean_test_score"]}).round(0)

> 📄 **Handout 3 — What Lasso keeps.** Circle the best alpha. Then find a simpler model within $500 of it.

<details><summary>Answer</summary>

* **Best: alpha = 300**, $24,222, keeping **105** of 237 columns
* **alpha = 1,000:** $24,510, only $288 worse, with **58** columns. When scores are this close, prefer the simpler model (Week 7)
* With every penalty, Lasso decides which columns matter. We didn't
</details>

In [ ]:
lasso = lasso_search.best_estimator_
lasso_coefs = pd.Series(lasso.named_steps["regressor"].coef_, index=lasso.named_steps["prep"].get_feature_names_out())
print("largest:", ", ".join(lasso_coefs.abs().sort_values(ascending=False).head(6).index.str.replace("num__|cat__", "", regex=True)))
print("dropped:", ", ".join(lasso_coefs[lasso_coefs == 0].index[:8].str.replace("num__|cat__", "", regex=True)), "...")

<details><summary>Answer</summary>

* Kept with the biggest slopes: living area, quality, year built, the Crawford neighborhood, lot area, basement area
* Dropped: `second_floor_area` (it's already inside `living_area`), `rooms`, `full_baths`, and most products
</details>

> "By inciting organizations to treat all data as useful, and by developing tools with a hitherto unmatched ability to find patterns, software engineers have created tools endowed with new and somewhat alien powers."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** We gave the model 237 columns and let Lasso pick which ones count. If an assessor's office valued your home this way, what would you want to know about the columns it kept and dropped?

<details><summary>Answer</summary>

* **Which columns could go in at all:** the neighborhood is in. In many U.S. cities, neighborhood lines still follow old redlining maps, so a neighborhood coefficient can carry that history into a price
* **Why a column was dropped:** "Lasso set it to 0" isn't a reason a homeowner can contest; with another split, other columns might survive
* Penalties tame wild coefficients. They don't decide what *should* count in a home's value. People do
</details>

Finally, once, the locked test set:

In [ ]:
for name, fitted in [("least squares, products", ols), ("Ridge, alpha = 300", ridge), ("Lasso, alpha = 300", lasso)]:
    rmse = np.sqrt(((fitted.predict(X_test) - y_test) ** 2).mean())
    print(f"{name:24} | test RMSE ${rmse:,.0f}")

> 📌 **One split:** on these 732 test houses all three land within $1,000 of each other ($21,172–$22,093). Cross-validation saw folds where least squares missed by $33,000; this test set happened to be kind to it. That's why we choose with cross-validation, not with one lucky split.

---
### Wrap-up

1. More columns (products) = more freedom → the line can **overfit**, and its coefficients can explode
2. **Ridge** charges α × Σ b²: every coefficient shrinks, none goes to 0
3. **Lasso** charges α × Σ |b|: many coefficients go to exactly 0, so it also chooses columns
4. Choose α with **cross-validation**; scale first; small α overfits, large α underfits 📌